# Tiempos totales programados y calibrados de las rutas QroBus

Este notebook calcula la duración completa de cada viaje usando `stop_times.txt`, resume P10, P50, promedio y P90 por ruta y sentido y reconstruye tiempos acumulados desde cada parada. Si existe el histórico local de Google, aprende una calibración robusta `constante + factor × GTFS` para producir escenarios P50 y P80. No realiza llamadas nuevas a Google Maps API.

> Como el GTFS local no contiene `shapes.txt`, el mapa une las paradas consecutivas del viaje más cercano a la mediana. Las líneas representan la secuencia de paradas, no la geometría exacta de las calles.


In [1]:
from pathlib import Path
import colorsys
import html
import sys

import folium
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = (Path("..") if Path("../data").exists() else Path(".")).resolve()
DATA_DIR = PROJECT_ROOT / "data" / "gtfs"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MAPS_DIR = PROJECT_ROOT / "maps"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
for directorio in [PROCESSED_DIR, MAPS_DIR]:
    directorio.mkdir(parents=True, exist_ok=True)
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from gtfs_network import gtfs_time_to_seconds

RESUMEN_PATH = PROCESSED_DIR / "tiempos_totales_rutas_gtfs.csv"
VIAJES_PATH = PROCESSED_DIR / "duraciones_viajes_gtfs.csv"
OBSERVACIONES_GOOGLE_PATH = PROCESSED_DIR / "google_maps_transit_observaciones.csv"
CALIBRACION_PATH = PROCESSED_DIR / "calibracion_tiempos_rutas_google.csv"
ACUMULADOS_PATH = PROCESSED_DIR / "tiempos_acumulados_paradas_rutas.csv"
MAPA_PATH = MAPS_DIR / "mapa_tiempos_totales_rutas_gtfs.html"
MIN_OBSERVACIONES_RUTA = 30
MIN_RANGO_GTFS_RUTA_MIN = 10
SEMIVIDA_OBSERVACIONES_DIAS = 14
print(f"GTFS: {DATA_DIR}")
print(f"Mapa: {MAPA_PATH}")


GTFS: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/data/gtfs
Mapa: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/maps/mapa_tiempos_totales_rutas_gtfs.html


## Funciones del análisis

Las funciones quedan definidas dentro del notebook para que el análisis no dependa de un script externo.


In [2]:
def calcular_duraciones_viajes(stop_times, trips, routes):
    """Devuelve una fila por viaje con su duración programada completa."""
    stop_times = stop_times.copy()
    stop_times["arrival_sec"] = gtfs_time_to_seconds(stop_times["arrival_time"])
    stop_times["departure_sec"] = gtfs_time_to_seconds(stop_times["departure_time"])
    stop_times["stop_sequence"] = pd.to_numeric(
        stop_times["stop_sequence"], errors="raise"
    )
    ordenados = stop_times.sort_values(["trip_id", "stop_sequence"])
    extremos = ordenados.groupby("trip_id", as_index=False).agg(
        stop_id_inicio=("stop_id", "first"),
        stop_id_fin=("stop_id", "last"),
        salida_inicio_sec=("departure_sec", "first"),
        llegada_fin_sec=("arrival_sec", "last"),
        numero_paradas=("stop_id", "size"),
    )
    extremos["duracion_programada_min"] = (
        extremos["llegada_fin_sec"] - extremos["salida_inicio_sec"]
    ) / 60
    invalidos = extremos["duracion_programada_min"].lt(0)
    if invalidos.any():
        ejemplos = extremos.loc[invalidos, "trip_id"].head().tolist()
        raise ValueError(f"Hay viajes con duración negativa: {ejemplos}")
    viajes = (
        extremos.merge(trips, on="trip_id", how="left", validate="one_to_one")
        .merge(routes, on="route_id", how="left", validate="many_to_one")
    )
    if viajes["route_id"].isna().any():
        raise ValueError("Hay trip_id de stop_times que no existen en trips.txt.")
    return viajes


def resumir_por_ruta_sentido(viajes):
    """Resume la variación programada sin ocultarla en un único promedio."""
    def percentil(q):
        return lambda serie: float(np.nanpercentile(serie, q))

    agrupadores = ["route_id", "route_short_name", "route_long_name", "direction_id"]
    resumen = viajes.groupby(
        agrupadores, dropna=False, as_index=False
    ).agg(
        numero_viajes=("trip_id", "nunique"),
        numero_servicios=("service_id", "nunique"),
        duracion_min_min=("duracion_programada_min", "min"),
        duracion_p10_min=("duracion_programada_min", percentil(10)),
        duracion_mediana_min=("duracion_programada_min", "median"),
        duracion_promedio_min=("duracion_programada_min", "mean"),
        duracion_p90_min=("duracion_programada_min", percentil(90)),
        duracion_max_min=("duracion_programada_min", "max"),
        paradas_mediana=("numero_paradas", "median"),
    )
    columnas_tiempo = [c for c in resumen if c.startswith("duracion_")]
    resumen[columnas_tiempo] = resumen[columnas_tiempo].round(2)
    resumen["paradas_mediana"] = resumen["paradas_mediana"].round(1)
    resumen["estado_calculo"] = "calculado"
    return resumen.sort_values(["route_short_name", "direction_id"])


def agregar_rutas_sin_viajes(resumen, routes):
    """Conserva el catálogo completo e identifica rutas sin horarios."""
    route_ids_con_viajes = set(resumen["route_id"].dropna().astype(str))
    faltantes = routes.loc[
        ~routes["route_id"].astype(str).isin(route_ids_con_viajes),
        ["route_id", "route_short_name", "route_long_name"],
    ].copy()
    if faltantes.empty:
        return resumen
    faltantes["direction_id"] = np.nan
    faltantes["numero_viajes"] = 0
    faltantes["numero_servicios"] = 0
    for columna in [
        "duracion_min_min", "duracion_p10_min", "duracion_mediana_min",
        "duracion_promedio_min", "duracion_p90_min", "duracion_max_min",
        "paradas_mediana",
    ]:
        faltantes[columna] = np.nan
    faltantes["estado_calculo"] = "sin viajes en trips.txt"
    return (
        pd.concat([resumen, faltantes[resumen.columns]], ignore_index=True)
        .sort_values(["route_short_name", "direction_id"], na_position="last")
        .reset_index(drop=True)
    )


### Calibración robusta con el histórico de Google

El modelo restringe la constante a valores no negativos y el factor a valores mayores o iguales que uno, porque su objetivo es representar atraso adicional y nunca reducir el horario GTFS. P50 y P80 se ajustan por separado mediante pérdida cuantílica y las observaciones recientes reciben mayor peso.


In [3]:
def percentil_ponderado(valores, pesos, q):
    """Calcula un percentil con pesos no negativos."""
    valores = np.asarray(valores, dtype=float)
    pesos = np.asarray(pesos, dtype=float)
    validos = np.isfinite(valores) & np.isfinite(pesos) & (pesos > 0)
    if not validos.any():
        return np.nan
    valores, pesos = valores[validos], pesos[validos]
    orden = np.argsort(valores)
    valores, pesos = valores[orden], pesos[orden]
    acumulado = np.cumsum(pesos)
    posicion = np.searchsorted(acumulado, q * acumulado[-1], side="left")
    return float(valores[min(posicion, len(valores) - 1)])


def preparar_observaciones_google(observaciones):
    """Limpia el histórico y calcula pesos de recencia sin llamar a la API."""
    requeridas = {
        "observed_at_utc", "route_id_principal", "tiempo_gtfs_min",
        "google_eta_min", "condition", "status_code",
    }
    faltantes = requeridas - set(observaciones.columns)
    if faltantes:
        raise ValueError(f"El histórico de Google no contiene: {sorted(faltantes)}")
    datos = observaciones.copy()
    datos["route_id_principal"] = (
        datos["route_id_principal"].astype("string").str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )
    datos["tiempo_gtfs_min"] = pd.to_numeric(datos["tiempo_gtfs_min"], errors="coerce")
    datos["google_eta_min"] = pd.to_numeric(datos["google_eta_min"], errors="coerce")
    datos["status_code"] = pd.to_numeric(datos["status_code"], errors="coerce")
    datos["observed_at_utc"] = pd.to_datetime(
        datos["observed_at_utc"], errors="coerce", utc=True, format="mixed"
    )
    datos = datos[
        datos["condition"].eq("ROUTE_EXISTS")
        & datos["status_code"].fillna(0).eq(0)
        & datos["tiempo_gtfs_min"].between(1, 180, inclusive="both")
        & datos["google_eta_min"].between(1, 180, inclusive="both")
        & datos["observed_at_utc"].notna()
    ].copy()
    razon = datos["google_eta_min"] / datos["tiempo_gtfs_min"]
    datos = datos[razon.between(0.25, 4, inclusive="both")].copy()
    if datos.empty:
        raise ValueError("No existen observaciones válidas para calibrar tiempos.")
    referencia = datos["observed_at_utc"].max()
    edad_dias = (referencia - datos["observed_at_utc"]).dt.total_seconds() / 86400
    datos["peso_recencia"] = 0.5 ** (edad_dias / SEMIVIDA_OBSERVACIONES_DIAS)
    return datos


def ajustar_modelo_cuantil(datos, q):
    """Ajusta Google = constante + factor × GTFS con restricciones de atraso."""
    x = datos["tiempo_gtfs_min"].to_numpy(dtype=float)
    y = datos["google_eta_min"].to_numpy(dtype=float)
    pesos = datos["peso_recencia"].to_numpy(dtype=float)
    limite_inferior, limite_superior = 1.0, 2.5
    mejor = None
    for _ in range(3):
        factores = np.linspace(limite_inferior, limite_superior, 301)
        for factor in factores:
            constante = max(0.0, percentil_ponderado(y - factor * x, pesos, q))
            error = y - (constante + factor * x)
            perdida = np.sum(
                pesos * np.where(error >= 0, q * error, (q - 1) * error)
            )
            if mejor is None or perdida < mejor[0]:
                mejor = (float(perdida), float(constante), float(factor))
        paso = (limite_superior - limite_inferior) / 300
        limite_inferior = max(1.0, mejor[2] - 2 * paso)
        limite_superior = min(2.5, mejor[2] + 2 * paso)
    return mejor[1], mejor[2]


def calibrar_modelos_por_ruta(observaciones):
    """Ajusta modelos por ruta y los contrae hacia un modelo global."""
    global_p50 = ajustar_modelo_cuantil(observaciones, 0.50)
    global_p80 = ajustar_modelo_cuantil(observaciones, 0.80)
    registros = [{
        "route_id": "__GLOBAL__",
        "observaciones_google": int(len(observaciones)),
        "rango_gtfs_observado_min": float(
            observaciones["tiempo_gtfs_min"].max()
            - observaciones["tiempo_gtfs_min"].min()
        ),
        "peso_modelo_ruta": 0.0,
        "constante_p50_min": global_p50[0],
        "factor_p50": global_p50[1],
        "constante_p80_min": max(global_p80[0], global_p50[0]),
        "factor_p80": max(global_p80[1], global_p50[1]),
        "fuente_modelo": "global",
    }]
    for route_id, grupo in observaciones.dropna(
        subset=["route_id_principal"]
    ).groupby("route_id_principal"):
        rango = float(grupo["tiempo_gtfs_min"].max() - grupo["tiempo_gtfs_min"].min())
        if len(grupo) < MIN_OBSERVACIONES_RUTA or rango < MIN_RANGO_GTFS_RUTA_MIN:
            continue
        ruta_p50 = ajustar_modelo_cuantil(grupo, 0.50)
        ruta_p80 = ajustar_modelo_cuantil(grupo, 0.80)
        peso = (len(grupo) / (len(grupo) + 100)) * min(1.0, rango / 20)
        constante_p50 = (1 - peso) * global_p50[0] + peso * ruta_p50[0]
        factor_p50 = (1 - peso) * global_p50[1] + peso * ruta_p50[1]
        constante_p80 = (1 - peso) * global_p80[0] + peso * ruta_p80[0]
        factor_p80 = (1 - peso) * global_p80[1] + peso * ruta_p80[1]
        registros.append({
            "route_id": str(route_id),
            "observaciones_google": int(len(grupo)),
            "rango_gtfs_observado_min": rango,
            "peso_modelo_ruta": float(peso),
            "constante_p50_min": float(constante_p50),
            "factor_p50": float(factor_p50),
            "constante_p80_min": float(max(constante_p80, constante_p50)),
            "factor_p80": float(max(factor_p80, factor_p50)),
            "fuente_modelo": "ruta suavizada con global",
        })
    return pd.DataFrame(registros)


In [4]:
def seleccionar_viajes_representativos(viajes, resumen):
    """Elige por ruta y sentido el viaje más cercano a la duración mediana."""
    calculados = resumen.loc[resumen["estado_calculo"].eq("calculado")].copy()
    calculados["direction_key"] = (
        pd.to_numeric(calculados["direction_id"], errors="coerce")
        .astype("Int64").astype("string").fillna("sin_sentido")
    )
    viajes_candidatos = viajes.copy()
    viajes_candidatos["direction_key"] = (
        pd.to_numeric(viajes_candidatos["direction_id"], errors="coerce")
        .astype("Int64").astype("string").fillna("sin_sentido")
    )
    columnas_resumen = [
        "route_id", "direction_key", "route_short_name", "route_long_name",
        "numero_viajes", "duracion_p10_min", "duracion_mediana_min",
        "duracion_p90_min", "constante_p50_min", "factor_p50",
        "constante_p80_min", "factor_p80",
        "tiempo_pasajero_p50_min", "tiempo_pasajero_p80_min",
        "observaciones_google", "fuente_modelo",
    ]
    candidatos = viajes_candidatos.merge(
        calculados[columnas_resumen],
        on=["route_id", "direction_key"], how="inner",
        validate="many_to_one", suffixes=("", "_resumen"),
    )
    candidatos["distancia_mediana_min"] = (
        candidatos["duracion_programada_min"]
        - candidatos["duracion_mediana_min"]
    ).abs()
    return (
        candidatos.sort_values(
            ["route_id", "direction_key", "distancia_mediana_min", "trip_id"]
        )
        .drop_duplicates(["route_id", "direction_key"], keep="first")
        .reset_index(drop=True)
    )


def construir_trazos_representativos(stop_times, stops, representativos):
    """Construye líneas aproximadas de los viajes representativos."""
    seleccion = representativos[
        [
            "trip_id", "route_id", "direction_key",
            "route_short_name_resumen", "route_long_name_resumen",
            "numero_viajes", "duracion_p10_min",
            "duracion_mediana_min", "duracion_p90_min",
            "constante_p50_min", "factor_p50",
            "constante_p80_min", "factor_p80",
            "tiempo_pasajero_p50_min", "tiempo_pasajero_p80_min",
            "observaciones_google", "fuente_modelo",
        ]
    ].copy()
    puntos = stop_times.loc[
        stop_times["trip_id"].isin(set(seleccion["trip_id"])),
        ["trip_id", "stop_id", "stop_sequence"],
    ].copy()
    puntos["stop_sequence"] = pd.to_numeric(
        puntos["stop_sequence"], errors="raise"
    )
    puntos = (
        puntos.merge(
            stops[["stop_id", "stop_name", "stop_lat", "stop_lon"]],
            on="stop_id", how="left", validate="many_to_one",
        )
        .merge(seleccion, on="trip_id", how="left", validate="many_to_one")
        .dropna(subset=["stop_lat", "stop_lon"])
        .sort_values(["trip_id", "stop_sequence"])
    )
    trazos = []
    for trip_id, grupo in puntos.groupby("trip_id", sort=False):
        primera = grupo.iloc[0]
        if len(grupo) < 2:
            continue
        trazos.append(
            {
                "trip_id": str(trip_id),
                "route_id": str(primera["route_id"]),
                "direction_key": str(primera["direction_key"]),
                "route_short_name": str(primera["route_short_name_resumen"]),
                "route_long_name": str(primera["route_long_name_resumen"]),
                "numero_viajes": int(primera["numero_viajes"]),
                "p10_min": float(primera["duracion_p10_min"]),
                "p50_min": float(primera["duracion_mediana_min"]),
                "p90_min": float(primera["duracion_p90_min"]),
                "estimado_p50_min": float(primera["tiempo_pasajero_p50_min"]),
                "estimado_p80_min": float(primera["tiempo_pasajero_p80_min"]),
                "observaciones_google": int(primera["observaciones_google"]),
                "fuente_modelo": str(primera["fuente_modelo"]),
                "coordenadas": grupo[["stop_lat", "stop_lon"]]
                    .astype(float).to_numpy().tolist(),
            }
        )
    return trazos


In [5]:
def construir_tiempos_acumulados(stop_times, stops, representativos):
    """Propaga segmentos corregidos hacia atrás desde el final de cada viaje."""
    columnas_modelo = [
        "trip_id", "route_id", "direction_key",
        "route_short_name_resumen", "constante_p50_min", "factor_p50",
        "constante_p80_min", "factor_p80", "observaciones_google",
        "fuente_modelo",
    ]
    seleccion = representativos[columnas_modelo].copy()
    puntos = stop_times.loc[
        stop_times["trip_id"].isin(set(seleccion["trip_id"])),
        ["trip_id", "stop_id", "stop_sequence", "arrival_time", "departure_time"],
    ].copy()
    puntos["stop_sequence"] = pd.to_numeric(puntos["stop_sequence"], errors="raise")
    puntos["arrival_sec"] = gtfs_time_to_seconds(puntos["arrival_time"])
    puntos["departure_sec"] = gtfs_time_to_seconds(puntos["departure_time"])
    puntos = (
        puntos.merge(seleccion, on="trip_id", how="inner", validate="many_to_one")
        .merge(stops[["stop_id", "stop_name"]], on="stop_id", how="left", validate="many_to_one")
        .sort_values(["trip_id", "stop_sequence"])
    )
    resultados = []
    for _, grupo in puntos.groupby("trip_id", sort=False):
        grupo = grupo.copy().reset_index(drop=True)
        siguiente_llegada = grupo["arrival_sec"].shift(-1)
        segmento_gtfs = ((siguiente_llegada - grupo["departure_sec"]) / 60).fillna(0)
        if not segmento_gtfs.iloc[:-1].between(0, 180, inclusive="both").all():
            raise ValueError(f"Segmentos inválidos en el viaje {grupo.iloc[0]['trip_id']}")
        factor_p50 = float(grupo.iloc[0]["factor_p50"])
        factor_p80 = float(grupo.iloc[0]["factor_p80"])
        segmento_p50 = segmento_gtfs * factor_p50
        segmento_p80 = segmento_gtfs * factor_p80
        restante_gtfs = segmento_gtfs.iloc[::-1].cumsum().iloc[::-1]
        restante_p50 = segmento_p50.iloc[::-1].cumsum().iloc[::-1]
        restante_p80 = segmento_p80.iloc[::-1].cumsum().iloc[::-1]
        hay_recorrido = segmento_gtfs.index < len(grupo) - 1
        grupo["next_stop_id"] = grupo["stop_id"].shift(-1)
        grupo["tiempo_segmento_gtfs_min"] = segmento_gtfs
        grupo["tiempo_segmento_p50_min"] = segmento_p50
        grupo["tiempo_segmento_p80_min"] = segmento_p80
        grupo["tiempo_restante_gtfs_min"] = restante_gtfs
        grupo["tiempo_restante_operativo_p50_min"] = restante_p50
        grupo["tiempo_restante_operativo_p80_min"] = restante_p80
        grupo["tiempo_restante_pasajero_p50_min"] = restante_p50 + np.where(
            hay_recorrido, float(grupo.iloc[0]["constante_p50_min"]), 0.0
        )
        grupo["tiempo_restante_pasajero_p80_min"] = restante_p80 + np.where(
            hay_recorrido, float(grupo.iloc[0]["constante_p80_min"]), 0.0
        )
        resultados.append(grupo)
    columnas = [
        "route_id", "route_short_name_resumen", "direction_key", "trip_id",
        "stop_sequence", "stop_id", "stop_name", "next_stop_id",
        "tiempo_segmento_gtfs_min", "tiempo_segmento_p50_min",
        "tiempo_segmento_p80_min", "tiempo_restante_gtfs_min",
        "tiempo_restante_operativo_p50_min",
        "tiempo_restante_operativo_p80_min",
        "tiempo_restante_pasajero_p50_min",
        "tiempo_restante_pasajero_p80_min",
        "constante_p50_min", "factor_p50", "constante_p80_min",
        "factor_p80", "observaciones_google", "fuente_modelo",
    ]
    return pd.concat(resultados, ignore_index=True)[columnas]


In [6]:
def crear_mapa_tiempos_totales(trazos, salida_html):
    """Dibuja una línea por ruta-sentido con GTFS y estimaciones P50/P80."""
    if not trazos:
        raise ValueError("No existen trazos calculables para crear el mapa.")
    todas_coords = [coord for trazo in trazos for coord in trazo["coordenadas"]]
    centro = [
        float(np.mean([coord[0] for coord in todas_coords])),
        float(np.mean([coord[1] for coord in todas_coords])),
    ]
    mapa = folium.Map(location=centro, zoom_start=11, tiles="OpenStreetMap")
    capa = folium.FeatureGroup(
        name=f"Rutas con tiempo GTFS y calibrado ({len(trazos)})", show=True
    )
    route_ids = sorted({str(trazo["route_id"]) for trazo in trazos})
    colores_ruta = {}
    for indice, route_id in enumerate(route_ids):
        tono = (indice * 0.618033988749895) % 1.0
        rojo, verde, azul = colorsys.hsv_to_rgb(tono, 0.72, 0.82)
        colores_ruta[route_id] = (
            f"#{round(rojo * 255):02x}{round(verde * 255):02x}{round(azul * 255):02x}"
        )
    elementos_leyenda = []
    for trazo in sorted(
        trazos, key=lambda x: (x["route_short_name"], x["direction_key"])
    ):
        color = colores_ruta[str(trazo["route_id"])]
        sentido = trazo["direction_key"].replace("sin_sentido", "sin dato")
        etiqueta = (
            f"Ruta {trazo['route_short_name']} · sentido {sentido} | "
            f"GTFS P50 {trazo['p50_min']:.1f} min | "
            f"estimado P50 {trazo['estimado_p50_min']:.1f} min | "
            f"estimado P80 {trazo['estimado_p80_min']:.1f} min | "
            f"Google n={trazo['observaciones_google']} ({trazo['fuente_modelo']})"
        )
        folium.PolyLine(
            trazo["coordenadas"], color=color, weight=4, opacity=0.78,
            tooltip=etiqueta,
        ).add_to(capa)
        elementos_leyenda.append(
            "<div style='white-space:nowrap;margin:2px 0'>"
            f"<span style='display:inline-block;width:14px;height:4px;"
            f"background:{color};margin-right:6px;vertical-align:middle'></span>"
            f"{html.escape(str(trazo['route_short_name']))} · S{html.escape(sentido)} "
            f"— GTFS {trazo['p50_min']:.1f} · "
            f"P50 {trazo['estimado_p50_min']:.1f} · "
            f"P80 {trazo['estimado_p80_min']:.1f} min</div>"
        )
    capa.add_to(mapa)
    folium.LayerControl(collapsed=False).add_to(mapa)
    mapa.fit_bounds(todas_coords)
    leyenda = folium.Element(
        "<div style='position:fixed;bottom:24px;left:24px;z-index:9999;"
        "background:white;border:1px solid #777;border-radius:5px;padding:8px;"
        "font-size:12px;max-height:300px;overflow-y:auto;box-shadow:0 1px 5px #999'>"
        "<b>Tiempo total GTFS y calibrado por ruta y sentido</b>"
        + "".join(elementos_leyenda)
        + "</div>"
    )
    mapa.get_root().html.add_child(leyenda)
    salida_html.parent.mkdir(parents=True, exist_ok=True)
    mapa.save(salida_html)
    return mapa


## 1. Carga del GTFS


In [7]:
stop_times = pd.read_csv(
    DATA_DIR / "stop_times.txt",
    dtype={"trip_id": "string", "stop_id": "string"},
)
trips = pd.read_csv(
    DATA_DIR / "trips.txt",
    dtype={"trip_id": "string", "route_id": "string"},
)
routes = pd.read_csv(DATA_DIR / "routes.txt", dtype={"route_id": "string"})
stops = pd.read_csv(DATA_DIR / "stops.txt", dtype={"stop_id": "string"})
print(f"Stop times: {len(stop_times):,}")
print(f"Viajes catalogados: {trips['trip_id'].nunique():,}")
print(f"Rutas catalogadas: {routes['route_id'].nunique():,}")


Stop times: 1,326,009
Viajes catalogados: 32,536
Rutas catalogadas: 217


## 2. Duración completa de cada viaje

Para cada `trip_id` se resta la salida de la primera parada a la llegada de la última. Las horas GTFS mayores a 24:00 se conservan correctamente.


In [8]:
viajes = calcular_duraciones_viajes(stop_times, trips, routes)
resumen = agregar_rutas_sin_viajes(
    resumir_por_ruta_sentido(viajes), routes
)
resumen.to_csv(RESUMEN_PATH, index=False)
viajes.to_csv(VIAJES_PATH, index=False)
print(f"Viajes analizados: {viajes['trip_id'].nunique():,}")
print(f"Rutas con viajes: {viajes['route_id'].nunique():,}")
print(
    "Rutas del catálogo sin viajes: "
    f"{resumen.loc[resumen['estado_calculo'].ne('calculado'), 'route_id'].nunique():,}"
)
display(
    resumen.loc[resumen["estado_calculo"].eq("calculado")].head(10)
)


Viajes analizados: 32,536
Rutas con viajes: 117
Rutas del catálogo sin viajes: 100


,route_id,route_short_name,route_long_name,direction_id,numero_viajes,numero_servicios,duracion_min_min,duracion_p10_min,duracion_mediana_min,duracion_promedio_min,duracion_p90_min,duracion_max_min,paradas_mediana,estado_calculo
42,386,AIQ01,Alameda - Aeropuerto Internacional De Queretaro,0.0,33,3,55.30,55.30,73.73,67.70,73.73,73.73,21.0,calculado
43,386,AIQ01,Alameda - Aeropuerto Internacional De Queretaro,1.0,33,3,52.63,52.63,70.02,63.70,70.02,70.02,28.0,calculado
44,96,C21,Las Flores - Fracc Fray Junipero Serra,0.0,204,3,93.22,93.22,104.88,102.89,109.30,109.30,69.0,calculado
45,96,C21,Las Flores - Fracc Fray Junipero Serra,1.0,204,3,93.73,93.73,98.28,98.95,103.87,103.87,64.0,calculado
46,97,C22,Fracc Santuarios - Penal,0.0,160,3,76.47,79.98,90.67,89.11,95.12,95.12,67.0,calculado
47,97,C22,Fracc Santuarios - Penal,1.0,160,3,79.73,85.18,94.78,93.83,101.60,101.60,70.0,calculado
48,98,C23,Lomas De La Cruz - Terminal De Autobuses (taq),0.0,188,3,91.78,91.78,98.15,96.74,99.87,99.87,69.0,calculado
49,98,C23,Lomas De La Cruz - Terminal De Autobuses (taq),1.0,188,3,80.47,80.47,90.95,88.05,91.85,91.85,65.0,calculado
50,319,C24,La Pradera - Bosques Del Cimatario,0.0,173,3,65.08,65.08,69.65,69.93,74.20,74.20,58.0,calculado
51,319,C24,La Pradera - Bosques Del Cimatario,1.0,175,3,60.77,60.77,73.45,69.54,73.45,73.45,51.0,calculado


## 3. Aprendizaje de la constante y el factor de atraso

Cada observación histórica compara una ETA de Google con el tiempo GTFS hacia Corregidora. Se ajustan dos modelos: `P50 = constante_P50 + factor_P50 × GTFS` y `P80 = constante_P80 + factor_P80 × GTFS`. La constante se aplica una sola vez por viaje y el factor se aplica a cada segmento.

La atribución por ruta usa `route_id_principal` del camino GTFS; Google no confirma en la matriz que haya utilizado esa misma línea. Por ello la calibración es un **proxy**. Las rutas con muestra insuficiente utilizan el modelo global y las demás se suavizan hacia él para reducir sobreajuste.


In [9]:
if OBSERVACIONES_GOOGLE_PATH.exists():
    observaciones_crudas = pd.read_csv(
        OBSERVACIONES_GOOGLE_PATH, dtype={"stop_id": "string"}
    )
    observaciones_modelo = preparar_observaciones_google(observaciones_crudas)
    calibracion = calibrar_modelos_por_ruta(observaciones_modelo)
else:
    observaciones_modelo = pd.DataFrame()
    calibracion = pd.DataFrame([{
        "route_id": "__GLOBAL__", "observaciones_google": 0,
        "rango_gtfs_observado_min": 0.0, "peso_modelo_ruta": 0.0,
        "constante_p50_min": 0.0, "factor_p50": 1.0,
        "constante_p80_min": 0.0, "factor_p80": 1.0,
        "fuente_modelo": "GTFS sin histórico de Google",
    }])

modelo_global = calibracion.loc[calibracion["route_id"].eq("__GLOBAL__")].iloc[0]
modelos_ruta = calibracion.loc[calibracion["route_id"].ne("__GLOBAL__")].copy()
resumen["route_id"] = resumen["route_id"].astype("string")
resumen = resumen.merge(modelos_ruta, on="route_id", how="left", validate="many_to_one")
for columna in ["constante_p50_min", "factor_p50", "constante_p80_min", "factor_p80"]:
    resumen[columna] = resumen[columna].fillna(float(modelo_global[columna]))
resumen["observaciones_google"] = resumen["observaciones_google"].fillna(0).astype(int)
resumen["rango_gtfs_observado_min"] = resumen["rango_gtfs_observado_min"].fillna(0.0)
resumen["peso_modelo_ruta"] = resumen["peso_modelo_ruta"].fillna(0.0)
resumen["fuente_modelo"] = resumen["fuente_modelo"].fillna("global")
resumen["tiempo_operativo_p50_min"] = (
    resumen["duracion_mediana_min"] * resumen["factor_p50"]
)
resumen["tiempo_operativo_p80_min"] = (
    resumen["duracion_mediana_min"] * resumen["factor_p80"]
)
resumen["tiempo_pasajero_p50_min"] = (
    resumen["constante_p50_min"] + resumen["tiempo_operativo_p50_min"]
)
resumen["tiempo_pasajero_p80_min"] = (
    resumen["constante_p80_min"] + resumen["tiempo_operativo_p80_min"]
)
columnas_estimadas = [
    "constante_p50_min", "factor_p50", "constante_p80_min", "factor_p80",
    "tiempo_operativo_p50_min", "tiempo_operativo_p80_min",
    "tiempo_pasajero_p50_min", "tiempo_pasajero_p80_min",
]
resumen[columnas_estimadas] = resumen[columnas_estimadas].round(2)
calibracion.to_csv(CALIBRACION_PATH, index=False)
resumen.to_csv(RESUMEN_PATH, index=False)
print(f"Observaciones válidas usadas: {len(observaciones_modelo):,}")
print(f"Rutas con calibración propia suavizada: {len(modelos_ruta):,}")
print(
    f"Modelo global P50: {modelo_global['constante_p50_min']:.2f} + "
    f"{modelo_global['factor_p50']:.3f} × GTFS"
)
print(
    f"Modelo global P80: {modelo_global['constante_p80_min']:.2f} + "
    f"{modelo_global['factor_p80']:.3f} × GTFS"
)
display(
    resumen.loc[resumen["estado_calculo"].eq("calculado"), [
        "route_short_name", "direction_id", "duracion_mediana_min",
        "tiempo_pasajero_p50_min", "tiempo_pasajero_p80_min",
        "observaciones_google", "fuente_modelo",
    ]].head(10)
)


Observaciones válidas usadas: 7,960
Rutas con calibración propia suavizada: 41
Modelo global P50: 0.00 + 1.000 × GTFS
Modelo global P80: 6.16 + 1.000 × GTFS


,route_short_name,direction_id,duracion_mediana_min,tiempo_pasajero_p50_min,tiempo_pasajero_p80_min,observaciones_google,fuente_modelo
42,AIQ01,0.0,73.73,73.73,79.89,0,global
43,AIQ01,1.0,70.02,70.02,76.18,0,global
44,C21,0.0,104.88,106.93,111.70,126,ruta suavizada con global
45,C21,1.0,98.28,100.33,105.10,126,ruta suavizada con global
46,C22,0.0,90.67,90.67,93.35,130,ruta suavizada con global
47,C22,1.0,94.78,94.78,97.46,130,ruta suavizada con global
48,C23,0.0,98.15,98.15,100.99,117,ruta suavizada con global
49,C23,1.0,90.95,90.95,93.79,117,ruta suavizada con global
50,C24,0.0,69.65,69.65,72.21,164,ruta suavizada con global
51,C24,1.0,73.45,73.45,76.01,164,ruta suavizada con global


## 4. Propagación por segmentos hacia la primera parada

Para cada ruta y sentido se elige el viaje más cercano a la mediana GTFS. Cada segmento se multiplica por su factor P50/P80 y luego se realiza una suma acumulada inversa desde la última parada. La constante se añade una sola vez al tiempo del pasajero; no se repite en cada segmento. Los segmentos corregidos son estimaciones modeladas: la matriz actual de Google entrega tiempos completos origen-destino, no observaciones independientes de cada segmento.


In [10]:
representativos = seleccionar_viajes_representativos(viajes, resumen)
tiempos_acumulados = construir_tiempos_acumulados(
    stop_times, stops, representativos
)
tiempos_acumulados.to_csv(ACUMULADOS_PATH, index=False)
assert tiempos_acumulados["tiempo_restante_operativo_p50_min"].ge(
    tiempos_acumulados["tiempo_restante_gtfs_min"] - 1e-9
).all()
assert tiempos_acumulados["tiempo_restante_pasajero_p80_min"].ge(
    tiempos_acumulados["tiempo_restante_pasajero_p50_min"] - 1e-9
).all()
print(f"Paradas acumuladas exportadas: {len(tiempos_acumulados):,}")
print(f"Archivo: {ACUMULADOS_PATH}")
display(tiempos_acumulados.head(10))


Paradas acumuladas exportadas: 8,071
Archivo: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/data/processed/tiempos_acumulados_paradas_rutas.csv


,route_id,route_short_name_resumen,direction_key,trip_id,stop_sequence,stop_id,stop_name,next_stop_id,tiempo_segmento_gtfs_min,tiempo_segmento_p50_min,...,tiempo_restante_operativo_p50_min,tiempo_restante_operativo_p80_min,tiempo_restante_pasajero_p50_min,tiempo_restante_pasajero_p80_min,constante_p50_min,factor_p50,constante_p80_min,factor_p80,observaciones_google,fuente_modelo
0,386,AIQ01,1,10,1,2033,Prol. Luis Pasteur/Constituyentes,3573,0.900000,0.900000,...,65.683333,65.683333,65.683333,71.843333,0.0,1.0,6.16,1.0,0,global
1,386,AIQ01,1,10,2,3573,Av. Luis Pateur Sur/José A. Bustamante,2034,0.366667,0.366667,...,64.783333,64.783333,64.783333,70.943333,0.0,1.0,6.16,1.0,0,global
2,386,AIQ01,1,10,3,2034,Prol. Luis Pasteu /Chichén Itzá,2035,0.450000,0.450000,...,64.416667,64.416667,64.416667,70.576667,0.0,1.0,6.16,1.0,0,global
3,386,AIQ01,1,10,4,2035,Prol. Luis Pasteur/José Sotelo,3341,1.416667,1.416667,...,63.966667,63.966667,63.966667,70.126667,0.0,1.0,6.16,1.0,0,global
4,386,AIQ01,1,10,5,3341,Carr. Lateral No. 57/Prol. del Cimatario,2751,0.766667,0.766667,...,62.550000,62.550000,62.550000,68.710000,0.0,1.0,6.16,1.0,0,global
5,386,AIQ01,1,10,6,2751,Lateral Querétaro A Mexico/Comercial Mexicana,3783,7.666667,7.666667,...,61.783333,61.783333,61.783333,67.943333,0.0,1.0,6.16,1.0,0,global
6,386,AIQ01,1,10,7,3783,San Isidro Miranda/Autopista México/Querétaro,3863,2.116667,2.116667,...,54.116667,54.116667,54.116667,60.276667,0.0,1.0,6.16,1.0,0,global
7,386,AIQ01,1,10,8,3863,Carr. Federal No. 57/Calle San Ignacio,5213,1.016667,1.016667,...,52.000000,52.000000,52.000000,58.160000,0.0,1.0,6.16,1.0,0,global
8,386,AIQ01,1,10,9,5213,Av. Estado de México/Av. Aguascalientes,3784,0.550000,0.550000,...,50.983333,50.983333,50.983333,57.143333,0.0,1.0,6.16,1.0,0,global
9,386,AIQ01,1,10,10,3784,Carr. Federal No. 57/Calle Jalisco,3864,1.100000,1.100000,...,50.433333,50.433333,50.433333,56.593333,0.0,1.0,6.16,1.0,0,global


## 5. Mapa por ruta y sentido

Cada ruta utiliza un color diferente. El tooltip y la leyenda comparan la mediana GTFS con los tiempos estimados P50 y P80 del pasajero e indican si el modelo procede de la ruta o del ajuste global.


In [11]:
representativos = seleccionar_viajes_representativos(viajes, resumen)
trazos = construir_trazos_representativos(stop_times, stops, representativos)
mapa_tiempos = crear_mapa_tiempos_totales(trazos, MAPA_PATH)
print(f"Combinaciones ruta-sentido dibujadas: {len(trazos):,}")
print(f"Copia HTML: {MAPA_PATH}")
mapa_tiempos


Combinaciones ruta-sentido dibujadas: 218
Copia HTML: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/maps/mapa_tiempos_totales_rutas_gtfs.html
